# 05 - predicted-structure arm (ESMFold)

Folds the WT once (HF `EsmForProteinFolding`, env `esm`), extracts 8 per-residue structural features (pLDDT, SASA, rel_SASA, burial, SSE helix/sheet/coil, contact number), and maps them to each variant by position. Reported as a separate predicted-structure tier, never mixed into the sequence-only results.

Per-variant delta features (dpLDDT/dSASA) need GPU folding of all 4,770 variants; see `src/colab_esmfold/09_colab_esmfold_extraction.ipynb` (runs locally in Jupyter, GPU strongly recommended, see notebook 06 for when to run it in the full pipeline).

In [ ]:
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/".projectroot").exists())
sys.path.insert(0, str(_root))
from paths import RAW, INTERIM, PROCESSED, FIGURES, TABLES, PROJECT_ROOT
SRC = _root / "src"


In [ ]:
import subprocess, sys
def run_in(env, script, *args):
    """Run src/<script> in conda env <env>; stream output; raise on failure."""
    cmd = ["conda","run","--no-capture-output","-n",env,"python",str(SRC/script), *map(str,args)]
    print(">>", " ".join(cmd))
    r = subprocess.run(cmd)
    if r.returncode != 0: raise RuntimeError(f"{script} failed in env {env} (rc={r.returncode})")


In [ ]:
if not (INTERIM/'esmfold_wt.pdb').exists(): run_in('esm','fold_wt.py')
else: print('cached WT fold')
run_in('esm','extract_struct.py')  # -> esmfold_wt_perres.npz
run_in('betalactam-v2','build_struct_features.py')  # -> feat_esmfold_struct.npz (position-mapped)